# 🎯 Handwritten Math Recognition with PaliGemma + LoRA

**Implementation of: "Representing Online Handwriting for Recognition in Large Vision-Language Models"**

This notebook trains a PaliGemma-3B model with LoRA on the MathWriting dataset for handwritten math recognition.

---

## 📋 Pipeline Overview
1. ✅ Setup environment & authenticate with HuggingFace
2. ✅ Download full MathWriting-2024 dataset
3. ✅ Upload project files (data_preprocessing.py, train.py, test_pipeline.py)
4. 🧪 **RUN SMOKE TEST** (validates pipeline before training)
5. ✅ Visualize InkML samples
6. ✅ Train PaliGemma with LoRA
7. ✅ Track training progress
8. ✅ Evaluate and visualize results


In [ ]:
# 🧪 INIT TEST: Catch Configuration Issues Early!
# ⚠️ RUN THIS FIRST to verify everything is set up correctly before proceeding
# This will check: HF auth, dataset paths, required files, GPU, and basic imports

import os
import sys

print("="*70)
print("🔍 INITIAL CONFIGURATION TEST")
print("="*70)
print("\nThis test verifies your Colab environment is properly configured.")
print("Run this BEFORE proceeding with training!\n")

# Track all checks
checks = {
    'colab_environment': False,
    'gpu_available': False,
    'hf_authentication': False,
    'dataset_path': False,
    'required_files': False,
    'python_imports': False
}

# 1. Check if running in Colab
try:
    import google.colab
    checks['colab_environment'] = True
    print("✅ Running in Google Colab")
except ImportError:
    print("⚠️  Not running in Colab - some features may not work")
    print("   (This is OK if running locally)")

# 2. Check GPU availability
try:
    import torch
    if torch.cuda.is_available():
        checks['gpu_available'] = True
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"✅ GPU available: {gpu_name} ({gpu_memory:.2f} GB VRAM)")
    else:
        print("⚠️  No GPU detected - training will be very slow on CPU")
except Exception as e:
    print(f"⚠️  Could not check GPU: {e}")

# 3. Check HuggingFace Authentication
print("\n" + "-"*70)
print("Checking HuggingFace Authentication...")
print("-"*70)

try:
    from huggingface_hub import whoami, login
    
    authenticated = False
    
    # Try Colab secrets first
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        if hf_token:
            login(token=hf_token, add_to_git_credential=False)
            user_info = whoami()
            print(f"✅ Authenticated via Colab secrets as: {user_info.get('name', 'user')}")
            authenticated = True
            checks['hf_authentication'] = True
    except ImportError:
        pass  # Not in Colab
    except Exception as e:
        print(f"⚠️  Colab secrets auth failed: {e}")
    
    # Try environment variable
    if not authenticated:
        hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
        if hf_token:
            try:
                login(token=hf_token, add_to_git_credential=False)
                user_info = whoami()
                print(f"✅ Authenticated via environment variable as: {user_info.get('name', 'user')}")
                authenticated = True
                checks['hf_authentication'] = True
            except Exception as e:
                print(f"⚠️  Environment variable auth failed: {e}")
    
    # Try existing CLI login
    if not authenticated:
        try:
            user_info = whoami()
            if user_info:
                print(f"✅ Using existing CLI login as: {user_info.get('name', 'user')}")
                authenticated = True
                checks['hf_authentication'] = True
        except Exception:
            pass
    
    if not authenticated:
        print("❌ NOT AUTHENTICATED")
        print("\n📝 To authenticate:")
        print("   1. Request access: https://huggingface.co/google/paligemma-3b-pt-224")
        print("   2. Create token: https://huggingface.co/settings/tokens")
        print("   3. Add to Colab secrets (🔑 icon): Name=HF_TOKEN, Value=your_token")
        print("   OR run: from huggingface_hub import notebook_login; notebook_login()")
        
except Exception as e:
    print(f"❌ Authentication check failed: {e}")

# 4. Check Dataset Path
print("\n" + "-"*70)
print("Checking Dataset Path...")
print("-"*70)

DATA_DIR = "/content/mathwriting-2024"  # Standard Colab path for full dataset

if os.path.exists(DATA_DIR):
    print(f"✅ Dataset directory exists: {DATA_DIR}")
    
    # Check for required splits
    required_splits = ['train', 'valid', 'test']
    missing_splits = []
    
    for split in required_splits:
        split_path = os.path.join(DATA_DIR, split)
        if os.path.exists(split_path):
            # Count .inkml files
            import glob
            inkml_files = glob.glob(os.path.join(split_path, "*.inkml"))
            if len(inkml_files) > 0:
                print(f"   ✅ {split}/: {len(inkml_files):,} .inkml files")
            else:
                print(f"   ⚠️  {split}/: exists but no .inkml files found")
                missing_splits.append(split)
        else:
            print(f"   ❌ {split}/: NOT FOUND")
            missing_splits.append(split)
    
    if len(missing_splits) == 0:
        checks['dataset_path'] = True
        print(f"\n✅ Dataset structure is correct!")
    else:
        print(f"\n⚠️  Missing splits: {', '.join(missing_splits)}")
        print(f"   Expected structure:")
        print(f"   {DATA_DIR}/")
        print(f"   ├── train/     (*.inkml files)")
        print(f"   ├── valid/     (*.inkml files)")
        print(f"   └── test/      (*.inkml files)")
else:
    print(f"❌ Dataset directory NOT FOUND: {DATA_DIR}")
    print(f"\n📝 To fix:")
    print(f"   1. Download MathWriting-2024 dataset")
    print(f"   2. Extract to: {DATA_DIR}")
    print(f"   3. Ensure structure: {DATA_DIR}/train/, {DATA_DIR}/valid/, {DATA_DIR}/test/")

# 5. Check Required Files
print("\n" + "-"*70)
print("Checking Required Project Files...")
print("-"*70)

REQUIRED_FILES = ['data_preprocessing.py', 'train.py', 'test_pipeline.py']
missing_files = []

for file in REQUIRED_FILES:
    file_path = f'/content/{file}'
    if os.path.exists(file_path):
        print(f"✅ {file}")
    else:
        print(f"❌ {file} - MISSING")
        missing_files.append(file)

if len(missing_files) == 0:
    checks['required_files'] = True
    print("\n✅ All required files present!")
    
    # Add to Python path
    if '/content' not in sys.path:
        sys.path.insert(0, '/content')
        print("✅ Added /content to Python path")
else:
    print(f"\n⚠️  Missing files: {', '.join(missing_files)}")
    print("📝 To fix:")
    print("   Upload files via Colab file browser (📂 icon on left)")
    print("   OR clone from GitHub: !git clone <repo_url> /content/realtime-math-1")

# 6. Test Python Imports
print("\n" + "-"*70)
print("Testing Python Imports...")
print("-"*70)

try:
    import torch
    import numpy as np
    from PIL import Image
    print("✅ Core libraries: torch, numpy, PIL")
    
    # Try importing our modules (if files exist)
    if checks['required_files']:
        try:
            from data_preprocessing import MathWritingDataset, InkMLParser
            print("✅ data_preprocessing.py imports successfully")
        except Exception as e:
            print(f"⚠️  data_preprocessing.py import failed: {e}")
        
        try:
            from train import MathTrainer
            print("✅ train.py imports successfully")
        except Exception as e:
            print(f"⚠️  train.py import failed: {e}")
        
        try:
            from test_pipeline import test_data_loading
            print("✅ test_pipeline.py imports successfully")
        except Exception as e:
            print(f"⚠️  test_pipeline.py import failed: {e}")
    
    checks['python_imports'] = True
except Exception as e:
    print(f"❌ Import test failed: {e}")

# Final Summary
print("\n" + "="*70)
print("📊 INIT TEST SUMMARY")
print("="*70)

critical_checks = ['hf_authentication', 'dataset_path', 'required_files']
all_critical_passed = all(checks[k] for k in critical_checks)

for check_name, passed in checks.items():
    status = "✅ PASS" if passed else "❌ FAIL"
    critical_marker = " [CRITICAL]" if check_name in critical_checks else ""
    print(f"  {check_name:20s}: {status}{critical_marker}")

if all_critical_passed:
    print("\n🎉 All critical checks PASSED!")
    print("✅ You're ready to proceed with training!")
    print(f"\n💡 Next steps:")
    print(f"   1. Run the environment setup cells below")
    print(f"   2. Run the smoke test cell (validates pipeline)")
    print(f"   3. Start training!")
else:
    print("\n⚠️  Some critical checks FAILED")
    print("❌ Please fix the issues above before proceeding")
    print("\n💡 Common fixes:")
    if not checks['hf_authentication']:
        print("   - Add HF_TOKEN to Colab secrets (🔑 icon)")
    if not checks['dataset_path']:
        print(f"   - Download dataset to: {DATA_DIR}")
    if not checks['required_files']:
        print("   - Upload required Python files via file browser")

print("="*70)

# Store DATA_DIR for use in other cells
DATA_DIR_COLAB = DATA_DIR


---
## 1️⃣ Environment Setup & Dependencies


In [ ]:
# Check GPU availability
!nvidia-smi

print("\n" + "="*70)
print("🚀 Starting Environment Setup")
print("="*70)


In [ ]:
# Install core dependencies
%pip install -q torch>=2.2.0 transformers>=4.50.0 peft>=0.15.0 datasets huggingface_hub
%pip install -q bitsandbytes>=0.45.0  # For quantization (optional but recommended)
%pip install -q matplotlib pillow numpy tqdm

print("\n✅ Dependencies installed!")


---
## 2️⃣ HuggingFace Authentication & Dataset Download


In [ ]:
from huggingface_hub import login, hf_hub_download, snapshot_download
import os

# Get HF token from Colab secrets
# Go to 🔑 (left sidebar) → Add new secret → Name: HF_TOKEN → Paste your token
# Create token at: https://huggingface.co/settings/tokens

print("="*70)
print("🔐 HuggingFace Authentication")
print("="*70)

authenticated = False

# Try Colab secrets first
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print("✅ Authenticated using HF_TOKEN from Colab secrets!")
    authenticated = True
except ImportError:
    print("ℹ️  Not running in Colab - trying alternative methods...")
except Exception as e:
    print(f"⚠️  Could not authenticate with Colab secrets: {e}")

# Fallback: Try notebook login
if not authenticated:
    try:
        print("\n📝 Please login manually:")
        from huggingface_hub import notebook_login
        notebook_login()
        authenticated = True
        print("✅ Manual authentication successful!")
    except Exception as e:
        print(f"❌ Manual authentication failed: {e}")

# Final check
if authenticated:
    print("\n🎉 Ready to download models from HuggingFace!")
else:
    print("\n⚠️  Warning: Authentication failed. You may not be able to download PaliGemma.")
    print("   Create a token at: https://huggingface.co/settings/tokens")
    print("   Then add it to Colab secrets (🔑 icon) as 'HF_TOKEN'")


In [ ]:
# Download MathWriting-2024 Full Dataset (2.9 GB)
import os
import subprocess

print("="*70)
print("📦 Downloading MathWriting-2024 Full Dataset")
print("="*70)

# Dataset configuration
data_dir = "/content/mathwriting-2024"
dataset_url = "https://storage.googleapis.com/mathwriting_data/mathwriting-2024.tgz"
download_path = "/content/mathwriting-2024.tgz"

# Check if dataset already exists
if os.path.exists(os.path.join(data_dir, "train")) and \
   os.path.exists(os.path.join(data_dir, "valid")) and \
   os.path.exists(os.path.join(data_dir, "test")):
    print("✅ Dataset already exists in /content/mathwriting-2024")
    
    # Count files
    import glob
    train_count = len(glob.glob(f"{data_dir}/train/*.inkml"))
    valid_count = len(glob.glob(f"{data_dir}/valid/*.inkml"))
    test_count = len(glob.glob(f"{data_dir}/test/*.inkml"))
    
    print(f"\n📊 Dataset Statistics:")
    print(f"   Train: {train_count:,} files")
    print(f"   Valid: {valid_count:,} files")
    print(f"   Test:  {test_count:,} files")
    print(f"   Total: {train_count + valid_count + test_count:,} files")
else:
    print(f"\n📥 Downloading full dataset from Google Cloud Storage...")
    print(f"   URL: {dataset_url}")
    print(f"   Size: ~2.9 GB (this may take 5-10 minutes)")
    print(f"   Destination: {data_dir}\n")
    
    try:
        # Download with progress
        print("⏳ Downloading...")
        result = subprocess.run(
            ["wget", "-q", "--show-progress", dataset_url, "-O", download_path],
            capture_output=False,
            text=True
        )
        
        if result.returncode == 0:
            print("\n✅ Download complete!")
            
            # Extract
            print(f"\n📦 Extracting to {data_dir}...")
            extract_result = subprocess.run(
                ["tar", "-xzf", download_path, "-C", "/content/"],
                capture_output=True,
                text=True
            )
            
            if extract_result.returncode == 0:
                print("✅ Extraction complete!")
                
                # Cleanup
                print("\n🧹 Cleaning up archive...")
                os.remove(download_path)
                
                # Verify structure
                print("\n✅ Verifying dataset structure...")
                splits = ['train', 'valid', 'test', 'symbols', 'synthetic']
                all_good = True
                
                for split in splits:
                    split_path = os.path.join(data_dir, split)
                    if os.path.exists(split_path):
                        import glob
                        inkml_files = glob.glob(os.path.join(split_path, "*.inkml"))
                        print(f"   ✅ {split:10s}: {len(inkml_files):,} .inkml files")
                    else:
                        print(f"   ⚠️  {split:10s}: NOT FOUND")
                        all_good = False
                
                if all_good:
                    print("\n🎉 Dataset ready for training!")
                else:
                    print("\n⚠️  Some splits are missing - please check extraction")
            else:
                print(f"❌ Extraction failed: {extract_result.stderr}")
        else:
            print(f"❌ Download failed!")
            print("\n📝 Manual download instructions:")
            print(f"   1. Download: {dataset_url}")
            print(f"   2. Upload to Colab via file browser")
            print(f"   3. Extract: !tar -xzf mathwriting-2024.tgz -C /content/")
            
    except Exception as e:
        print(f"❌ Error during download/extraction: {e}")
        print("\n📝 Manual download instructions:")
        print(f"   1. Download: {dataset_url}")
        print(f"   2. Upload to Colab via file browser")
        print(f"   3. Extract: !tar -xzf mathwriting-2024.tgz -C /content/")

print("\n" + "="*70)

In [ ]:
# Uncomment if you want to use Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# 
# # Update with your Drive path
# data_dir = "/content/drive/MyDrive/mathwriting-2024"
# print(f"✅ Using dataset from: {data_dir}")


---
## 3️⃣ Upload Project Files


In [ ]:
# Verify uploaded files
import os

required_files = ['data_preprocessing.py', 'train.py', 'test_pipeline.py']
missing_files = []

print("="*70)
print("📋 Checking Required Files")
print("="*70)

for file in required_files:
    if os.path.exists(f'/content/{file}'):
        print(f"✅ {file}")
    else:
        print(f"❌ {file} - MISSING")
        missing_files.append(file)

if missing_files:
    print(f"\n⚠️  Missing files: {', '.join(missing_files)}")
    print("Please upload them using the file browser (📂 icon)")
else:
    print("\n🎉 All required files present!")
    
# Add current directory to Python path
import sys
if '/content' not in sys.path:
    sys.path.insert(0, '/content')


In [ ]:
# Clone project files from GitHub (recommended)
print("="*70)
print("📁 Fetching Project Files from GitHub")
print("="*70)

import os

# Clone repository and copy Python files
if not os.path.exists('/content/data_preprocessing.py'):
    print("\n📥 Cloning repository...")
    !git clone https://github.com/hudsonmp/realtime-math.git /content/realtime-math-temp
    
    # Copy required Python files
    print("📋 Copying Python files to /content/...")
    !cp /content/realtime-math-temp/*.py /content/ 2>/dev/null || true
    
    # Cleanup
    !rm -rf /content/realtime-math-temp
    
    print("\n✅ Files copied successfully!")
else:
    print("✅ Files already present!")

# Verify files
print("\n📋 Verifying Required Files:")
required_files = ['data_preprocessing.py', 'train.py', 'test_pipeline.py']
for file in required_files:
    if os.path.exists(f'/content/{file}'):
        print(f"   ✅ {file}")
    else:
        print(f"   ❌ {file} - MISSING")

# Add to Python path
import sys
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
    print("\n✅ Added /content to Python path")

# Create directory structure
!mkdir -p /content/checkpoints
!mkdir -p /content/visualizations
print("✅ Directory structure created!")

print("\n" + "="*70)

In [ ]:
# 🧪 Pre-Training Test (Run Before Full Training!)
# ⚠️ IMPORTANT: Run this test BEFORE starting the full training to catch any issues early!

import sys
import os
import torch

# Ensure test_pipeline is available
if not os.path.exists('/content/test_pipeline.py'):
    print("❌ test_pipeline.py not found!")
    print("Please upload it via the file browser.")
else:
    # Update data_dir in test_pipeline to use Colab path
    data_dir = "/content/mathwriting-2024"
    
    print("="*70)
    print("🧪 RUNNING SMOKE TEST")
    print("="*70)
    print("\nThis will validate your pipeline with a small subset of data.")
    print("Expected runtime: ~2-3 minutes")
    print("\n" + "="*70 + "\n")
    
    # Import and run test
    try:
        # Temporarily modify sys.argv to pass data_dir
        original_argv = sys.argv
        sys.argv = ['test_pipeline.py']
        
        # Import test functions
        from test_pipeline import (
            test_data_loading,
            test_model_initialization, 
            test_training_steps,
            test_validation
        )
        
        # Run all tests
        results = {}
        
        # Test 1: Data loading
        results['data_loading'] = test_data_loading(data_dir)
        if not results['data_loading']:
            print("\n❌ CRITICAL: Data loading failed. Fix this before continuing.")
            sys.argv = original_argv
            raise SystemExit(1)
        
        # Test 2: Model initialization
        trainer, device = test_model_initialization(
            data_dir=data_dir,
            device="cuda" if torch.cuda.is_available() else "cpu"
        )
        results['model_init'] = trainer is not None
        if not results['model_init']:
            print("\n❌ CRITICAL: Model initialization failed. Fix this before continuing.")
            sys.argv = original_argv
            raise SystemExit(1)
        
        # Test 3: Training steps
        results['training'] = test_training_steps(trainer, device, num_steps=2)
        
        # Test 4: Validation
        results['validation'] = test_validation(trainer, device)
        
        # Summary
        print("\n" + "="*70)
        print("📊 TEST SUMMARY")
        print("="*70)
        for test_name, passed in results.items():
            status = "✅ PASS" if passed else "❌ FAIL"
            print(f"  {test_name:20s}: {status}")
        
        all_passed = all(results.values())
        if all_passed:
            print("\n🎉 All tests PASSED! Safe to proceed with full training.")
        else:
            print("\n⚠️  Some tests FAILED. Review errors above before training.")
        
        print("="*70)
        sys.argv = original_argv
        
    except Exception as e:
        print(f"\n❌ Test execution failed: {e}")
        import traceback
        traceback.print_exc()
        sys.argv = original_argv


---
## 4️⃣ Visualize InkML Samples

**Note:** If all smoke tests passed above ✅, you're ready to proceed!

Let's visualize some samples from the dataset to understand the data!


In [ ]:
# Import our preprocessing modules
from data_preprocessing import InkMLParser, StrokeNormalizer, InkRenderer
import matplotlib.pyplot as plt
import glob
import numpy as np

data_dir = "/content/mathwriting-2024"

# Get sample files
train_files = sorted(glob.glob(f"{data_dir}/train/*.inkml"))[:6]

if not train_files:
    print("❌ No InkML files found. Please check dataset path.")
else:
    print(f"✅ Found {len(train_files)} sample files to visualize\n")
    
    # Initialize components
    parser = InkMLParser()
    normalizer = StrokeNormalizer(target_points_per_stroke=16, N=224)
    renderer = InkRenderer(image_size=224, num_lines=2)
    
    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle('MathWriting Dataset Samples (Rendered with Time+Distance Info)', 
                 fontsize=16, fontweight='bold')
    
    for idx, (ax, file_path) in enumerate(zip(axes.flat, train_files)):
        # Parse and normalize
        strokes, label = parser.parse_file(file_path)
        normalized = normalizer.normalize(strokes)
        
        # Render image
        image = renderer.render(normalized)
        
        # Display
        ax.imshow(image)
        ax.set_title(f"Label: {label}\n({len(strokes)} strokes)", fontsize=10)
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('/content/visualizations/sample_renders.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n✅ Visualization saved to /content/visualizations/sample_renders.png")


In [ ]:
# Import training components
import torch
from train import MathTrainer
from data_preprocessing import MathWritingDataset

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n🖥️  Using device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


In [ ]:
# Load datasets
data_dir = "/content/mathwriting-2024"

print("\n" + "="*70)
print("📊 Loading Datasets")
print("="*70)

try:
    train_ds = MathWritingDataset(data_dir, split='train')
    valid_ds = MathWritingDataset(data_dir, split='valid')
    test_ds = MathWritingDataset(data_dir, split='test')
    
    print(f"\n✅ Datasets loaded successfully!")
    print(f"   Train: {len(train_ds):,} samples")
    print(f"   Valid: {len(valid_ds):,} samples")
    print(f"   Test:  {len(test_ds):,} samples")
    print(f"   Total: {len(train_ds) + len(valid_ds) + len(test_ds):,} samples")
    
except Exception as e:
    print(f"\n❌ Error loading datasets: {e}")
    print("\nPlease ensure the dataset is properly downloaded to:")
    print(f"   {data_dir}")


In [ ]:
# Initialize trainer with PaliGemma + LoRA
print("\n" + "="*70)
print("🤖 Initializing PaliGemma-3B with LoRA")
print("="*70)

trainer = MathTrainer(
    data_dir=data_dir,
    model_name="google/paligemma-3b-pt-224",
    output_dir="/content/checkpoints",
    device=device
)

print("\n✅ Model initialized successfully!")
print("\n📊 Trainable Parameters:")
trainer.model.print_trainable_parameters()


In [ ]:
---
## 6️⃣ Training with Progress Tracking

Let's train the model! This will take several hours depending on your GPU.


---
## 5️⃣ Train PaliGemma with LoRA

**🚀 Ready to train!** This will fine-tune PaliGemma-3B with LoRA on the full MathWriting dataset.
